In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

In [2]:
my_string = "the cat sat on the mat"
vocab = ["<pad>", "<unk>", "the", "cat", "sat", "on", "mat"]

T = len(my_string.split())
vocab_size = len(vocab)
d_m = 4
d_k = 4
d_v = 4
d_ff = 4 * d_m


In [3]:
vocab = ["<pad>", "<unk>", "the", "cat", "sat", "on", "mat"]

token_to_id = {token : id for id, token in enumerate(vocab)}
id_to_token = {id : token for token, id in token_to_id.items()}

tokens = my_string.split()
# tokens = [token_to_id[token.lower()] for token in my_string.split()]
token_ids = torch.tensor([token_to_id.get(token.lower(), "<unk>") for token in tokens])

In [4]:
tokens, token_ids

(['the', 'cat', 'sat', 'on', 'the', 'mat'], tensor([2, 3, 4, 5, 2, 6]))

In [5]:
x = token_ids

In [6]:
E = torch.eye(vocab_size, d_m)

In [7]:
E

tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

In [8]:
positional_enc = torch.zeros((T, d_m))
positional_enc[:, 0] = torch.arange(T)

In [9]:
positional_enc

tensor([[0., 0., 0., 0.],
        [1., 0., 0., 0.],
        [2., 0., 0., 0.],
        [3., 0., 0., 0.],
        [4., 0., 0., 0.],
        [5., 0., 0., 0.]])

In [10]:
x1 = E[token_ids] + positional_enc

In [11]:
x1

tensor([[0., 0., 1., 0.],
        [1., 0., 0., 1.],
        [2., 0., 0., 0.],
        [3., 0., 0., 0.],
        [4., 0., 1., 0.],
        [5., 0., 0., 0.]])

In [12]:
torch.softmax(x1, dim=0)

tensor([[0.0043, 0.1667, 0.2881, 0.1296],
        [0.0116, 0.1667, 0.1060, 0.3522],
        [0.0315, 0.1667, 0.1060, 0.1296],
        [0.0858, 0.1667, 0.1060, 0.1296],
        [0.2331, 0.1667, 0.2881, 0.1296],
        [0.6337, 0.1667, 0.1060, 0.1296]])

In [13]:
W_Q = torch.randn((d_m, d_k))
W_K = torch.randn((d_m, d_k))
W_V = torch.randn((d_m, d_k))

In [14]:
Q = x1 @ W_Q
K = x1 @ W_K
V = x1 @ W_V

In [15]:
mask = torch.triu(torch.ones((T, T)) * -torch.inf, diagonal=1)
A = torch.softmax((Q @ K.T) / math.sqrt(d_k) + mask, dim=1) @ V

In [16]:
A.shape

torch.Size([6, 4])

In [17]:
A

tensor([[-1.4137, -0.9507, -0.2977, -0.3765],
        [-0.4874,  5.1561,  0.2301, -0.4165],
        [-0.5932,  5.4306,  0.1863, -0.5073],
        [-0.5422,  5.5218,  0.2150, -0.4814],
        [-0.4717,  5.4795,  0.2438, -0.4377],
        [-0.4527,  5.4720,  0.2518, -0.4261]])

In [18]:
x2 = A + x1

In [19]:
def layer_norm(x: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    x_norm = (x - mean) / torch.sqrt(var + eps)
    return x_norm

In [20]:
x3 = layer_norm(x2)

In [21]:
W1 = torch.randn((d_m, d_ff))
W2 = torch.randn((d_ff, d_m))

b1 = torch.randn((1, d_ff))
b2 = torch.randn((1, d_m))

In [22]:
h = torch.relu(x3 @ W1 + b1)
x4 = h @ W2 + b2

In [23]:
x4

tensor([[-4.7130,  3.9173, -1.4397, -5.3564],
        [ 5.6722, -2.9727, -8.2492, -8.0756],
        [ 3.6246, -1.6743, -8.4409, -7.3829],
        [ 2.8531, -1.7543, -8.1994, -6.9284],
        [ 0.8950, -0.3257, -7.4285, -5.9649],
        [ 0.6483, -1.6709, -7.4128, -5.9765]])

In [24]:
x5 = x4 + x3
x6 = layer_norm(x5)

In [25]:
x6

tensor([[-1.0388,  1.3794,  0.5226, -0.8632],
        [ 1.4703,  0.3724, -0.9512, -0.8915],
        [ 1.3011,  0.6411, -1.0415, -0.9008],
        [ 1.2954,  0.6466, -1.0667, -0.8752],
        [ 1.0377,  0.9582, -1.0789, -0.9171],
        [ 1.2167,  0.7452, -1.1221, -0.8398]])

In [26]:
W_out = torch.randn((d_m, vocab_size))
b_out = torch.randn((1, vocab_size))

In [27]:
logits = x6 @ W_out + b_out

In [28]:
logits

tensor([[-4.1998, -3.7198,  0.8217, -0.9763, -0.0205, -0.8126,  2.6797],
        [-0.9248, -1.3525,  0.9233, -0.4167, -1.7152,  2.1159,  0.7433],
        [-0.9136, -1.8274,  0.9686, -0.6516, -1.7695,  2.1350,  1.0601],
        [-0.8478, -1.8338,  0.9583, -0.6434, -1.7681,  2.1713,  1.0829],
        [-1.0335, -2.4090,  1.0159, -0.9214, -1.7692,  2.0439,  1.4550],
        [-0.7691, -2.0073,  0.9563, -0.7080, -1.7708,  2.2109,  1.2288]])

In [29]:
probs = torch.softmax(logits, dim=1)

In [30]:
probs

tensor([[0.0008, 0.0013, 0.1217, 0.0201, 0.0524, 0.0237, 0.7799],
        [0.0275, 0.0179, 0.1747, 0.0457, 0.0125, 0.5757, 0.1459],
        [0.0263, 0.0106, 0.1729, 0.0342, 0.0112, 0.5552, 0.1895],
        [0.0274, 0.0102, 0.1670, 0.0337, 0.0109, 0.5616, 0.1891],
        [0.0225, 0.0057, 0.1750, 0.0252, 0.0108, 0.4893, 0.2715],
        [0.0283, 0.0082, 0.1586, 0.0300, 0.0104, 0.5562, 0.2083]])

In [31]:
torch.argmax(probs, dim=1)

tensor([6, 5, 5, 5, 5, 5])

In [32]:
h = 1
d_m = 16
d_k = int(d_m / h)
d_v = int(d_m / h)

T = 32

In [33]:
X = torch.randn((T, d_m))

W_Q = torch.randn((d_m, d_k))
W_K = torch.randn((d_m, d_k))
W_V = torch.randn((d_m, d_v))

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

In [34]:
print("X shape: ", X.shape)
print("W_Q shape: ", W_Q.shape)
print("W_K shape: ", W_K.shape)
print("W_V shape: ", W_V.shape)
print("Q shape: ", Q.shape)
print("K shape: ", K.shape)
print("V shape: ", V.shape)

X shape:  torch.Size([32, 16])
W_Q shape:  torch.Size([16, 16])
W_K shape:  torch.Size([16, 16])
W_V shape:  torch.Size([16, 16])
Q shape:  torch.Size([32, 16])
K shape:  torch.Size([32, 16])
V shape:  torch.Size([32, 16])


In [35]:
h = 1
d_m = 16
d_k = int(d_m / h)
d_v = int(d_m / h)

T = 32

X = torch.randn((T, d_m))

matrices = {}
qkvs = {}
attentions = []
for i in range(h):
    matrices[f"W_Q_{i}"] = torch.randn((d_m, d_k))
    matrices[f"W_K_{i}"] = torch.randn((d_m, d_k))
    matrices[f"W_V_{i}"] = torch.randn((d_m, d_v))

    Q = X @ matrices[f"W_Q_{i}"]
    K = X @ matrices[f"W_K_{i}"]
    V = X @ matrices[f"W_V_{i}"]
    qkvs[f"Q_{i}"] = Q
    qkvs[f"K_{i}"] = K
    qkvs[f"V_{i}"] = V

    mask = torch.triu(torch.ones((T, T)) * -torch.inf, diagonal=1)
    A = torch.softmax((Q @ K.T) / math.sqrt(d_k) + mask, dim=1) @ V
    attentions.append(A)

out = torch.tensor(attentions[0])
for i in range(1, len(attentions)):
    out = torch.cat((out, attentions[i]), dim=1)

W_O = torch.randn((d_m, d_m))
out = out @ W_O



/tmp/ipykernel_53139/3683660547.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  out = torch.tensor(attentions[0])


In [36]:
out = torch.tensor(attentions[0])
for i in range(1, len(attentions)):
    out = torch.cat((out, attentions[i]), dim=1)

/tmp/ipykernel_53139/2237657989.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  out = torch.tensor(attentions[0])


In [37]:
out.shape

torch.Size([32, 16])

In [38]:
h = 1
d_m = 16
d_k = int(d_m / h)
d_v = int(d_m / h)

T = 32

X = torch.randn((T, d_m))

W_Q = torch.randn((h, T, d_k))
W_K = torch.randn((h, T, d_k))
W_V = torch.randn((h, T, d_v))




In [41]:
a = torch.tensor([
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15]
], dtype=torch.float)
b = torch.eye(5)

b = torch.randn((4, 5, ), dtype=torch.float)

In [42]:
a.shape, b.shape

(torch.Size([3, 5]), torch.Size([4, 5]))

In [45]:
(a @ b)[0, :, :]

RuntimeError: mat1 and mat2 shapes cannot be multiplied (3x5 and 4x5)

In [46]:
a = torch.randn(20, 16)
b = torch.randn(4, 16, 4)
c = a @ b

a.shape, b.shape, c.shape

(torch.Size([20, 16]), torch.Size([4, 16, 4]), torch.Size([4, 20, 4]))

In [47]:
T = 20
d_m = 16
h = 4

In [48]:
Q = torch.randn((T, d_m))

In [49]:
Q.reshape((h, T, int(d_m / h))).shape

torch.Size([4, 20, 4])

In [50]:
Q.reshape((T, h, int(d_m / h))).permute(1, 0, 2).shape

torch.Size([4, 20, 4])

In [51]:
K = Q.clone()

In [52]:
Q == K

tensor([[True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,

In [53]:
Q.reshape((h, T, int(d_m / h))) == Q.reshape((T, h, int(d_m / h))).permute(1, 0, 2)

tensor([[[ True,  True,  True,  True],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False]],

        [[False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False,

In [54]:
# benchmark tensor heads vs one big matrix

# nvm i dont have a gpu lol

In [56]:
import os
import requests
import tiktoken
import numpy as np
import random

In [57]:
input_file_path = './data/tinyshakespeare/input.txt'

with open(input_file_path, 'r', encoding='utf-8') as f:
    data = f.read()
n = len(data)
train_data = data[:int(n*0.9)]
val_data = data[int(n*0.9):]

enc = tiktoken.get_encoding('gpt2')
train_ids = torch.tensor(enc.encode_ordinary(train_data), dtype=torch.long)
val_ids = torch.tensor(enc.encode_ordinary(val_data), dtype=torch.long)
print(f"train tokens: {len(train_ids):,}")
print(f"val tokens: {len(val_ids):,}")

train tokens: 301,966
val tokens: 36,059


In [58]:
train_ids[2]

tensor(25)

In [59]:
len(enc._mergeable_ranks) + len(enc._special_tokens), enc.n_vocab

(50257, 50257)

In [60]:
T = block_size = 32
vocab_size = enc.n_vocab
L = n_layer = 3
h = n_head = 4
d_m = n_embd = 96
d_k = int(d_m / h)
d_v = int(d_m / h)
d_ff = 384  # 4 * d_m
batch_size = 8
# dropout = 0.1

In [61]:
E = torch.randn((vocab_size, d_m), requires_grad=True)
W_Q = torch.randn((d_m, d_k * h), requires_grad=True)  # big ass matrix so multiply it back
W_K = torch.randn((d_m, d_k * h), requires_grad=True)
W_V = torch.randn((d_m, d_v * h), requires_grad=True)
M = torch.triu(torch.ones((T, T)) * -torch.inf, diagonal=1)
W_1 = torch.randn((d_m, d_ff), requires_grad=True)
b_1 = torch.randn((1, d_ff), requires_grad=True)
W_2 = torch.randn((d_ff, d_m), requires_grad=True)
b_2 = torch.randn((1, d_m), requires_grad=True)
W_O = torch.randn((d_m, vocab_size), requires_grad=True)
b_O = torch.randn((1, vocab_size), requires_grad=True)

In [ ]:
for e in range(5):
    Xs = []
    Ys = []
    for _ in range(batch_size):
        i = random.randrange(len(train_ids) - T)
        x = train_ids[i : i + T]
        y = train_ids[i + 1 : i + T + 1]
        Xs.append(x)
        Ys.append(y)
    X = torch.stack(Xs)
    print(f"X = {X.shape}")
    Y = torch.stack(Ys)  # (B, T)
    print(f"Y = {Y.shape}")
    # forward
    X1 = E[X]
    print(f"Emb = {X1.shape}")
    # positional encoding
    Q = X1 @ W_Q
    K = X1 @ W_K
    V = X1 @ W_V
    print(f"Q = {Q.shape}")
    print(f"K = {K.shape}")
    print(f"V = {V.shape}")
    A = torch.softmax((Q @ K.transpose(-2, -1)) / math.sqrt(d_k) + M, dim=-1) @ V
    print(f"A = {A.shape}")
    X2 = A + X1
    X3 = (X2 - X2.mean(dim=-1, keepdim=True)) / torch.sqrt(X2.var(dim=-1, keepdim=True, unbiased=False) + 1e-5)
    X4 = torch.relu(X3 @ W_1 + b_1)
    X5 = X4 @ W_2 + b_2
    X6 = X5 + X3
    X7 = (X6 - X6.mean(dim=-1, keepdim=True)) / torch.sqrt(X6.var(dim=-1, keepdim=True, unbiased=False) + 1e-5)
    logits = X7 @ W_O + b_O
    probs = torch.softmax(logits, dim=-1)
    loss = -torch.log(probs.gather(dim=-1, index=Y.unsqueeze(-1)).squeeze(-1)).mean()
    print(f"loss: {loss.item()}")

    # backward
    for param in (E, W_Q, W_K, W_V, W_1, b_1, W_2, b_2, W_O, b_O):
        param.grad = None
    loss.backward()

    # update
    lr = 0.08
    for param in (E, W_Q, W_K, W_V, W_1, b_1, W_2, b_2, W_O, b_O):
        param.data -= lr * param.grad

X = torch.Size([8, 32])
Y = torch.Size([8, 32])
Emb = torch.Size([8, 32, 96])
Q = torch.Size([8, 32, 96])
K = torch.Size([8, 32, 96])
V = torch.Size([8, 32, 96])
A = torch.Size([8, 32, 96])
loss: 42.155914306640625
X = torch.Size([8, 32])
Y = torch.Size([8, 32])
Emb = torch.Size([8, 32, 96])
Q = torch.Size([8, 32, 96])
K = torch.Size([8, 32, 96])
V = torch.Size([8, 32, 96])
A = torch.Size([8, 32, 96])
loss: 42.41567611694336
X = torch.Size([8, 32])
Y = torch.Size([8, 32])
Emb = torch.Size([8, 32, 96])
Q = torch.Size([8, 32, 96])
K = torch.Size([8, 32, 96])
V = torch.Size([8, 32, 96])
A = torch.Size([8, 32, 96])
loss: 42.22826385498047
X = torch.Size([8, 32])
Y = torch.Size([8, 32])
Emb = torch.Size([8, 32, 96])
Q = torch.Size([8, 32, 96])
K = torch.Size([8, 32, 96])
V = torch.Size([8, 32, 96])
A = torch.Size([8, 32, 96])
loss: 42.317420959472656
X = torch.Size([8, 32])
Y = torch.Size([8, 32])
Emb = torch.Size([8, 32, 96])
Q = torch.Size([8, 32, 96])
K = torch.Size([8, 32, 96])
V = torc

In [68]:
K.shape, K.T.shape, K.transpose(1, 2).shape

(torch.Size([8, 32, 96]), torch.Size([96, 32, 8]), torch.Size([8, 96, 32]))

In [94]:
tokens = [enc.decode([int(i)]) for i in probs.argmax(dim=1)]

In [95]:
tokens

[' Citizen',
 ':',
 '\n',
 '\n',
 ' we',
 ' proceed',
 ' any',
 ' further',
 ',',
 ' speak',
 ' me',
 ' speak',
 '.',
 '\n',
 '\n',
 '\n',
 ':',
 '\n',
 '\n',
 'ak',
 ',',
 ' speak',
 '.',
 '\n',
 '\n',
 '\n',
 ' Citizen',
 ':',
 '\n',
 '\n',
 ' are',
 ' all']

In [90]:
X

tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597,  2252,    11,
         3285,   502,  2740,    13,   198,   198,  3237,    25,   198,  5248,
          461,    11,  2740,    13,   198,   198,  5962, 22307,    25,   198,
         1639,   389])

In [91]:
[enc.decode([int(i)]) for i in X]

['First',
 ' Citizen',
 ':',
 '\n',
 'Before',
 ' we',
 ' proceed',
 ' any',
 ' further',
 ',',
 ' hear',
 ' me',
 ' speak',
 '.',
 '\n',
 '\n',
 'All',
 ':',
 '\n',
 'Spe',
 'ak',
 ',',
 ' speak',
 '.',
 '\n',
 '\n',
 'First',
 ' Citizen',
 ':',
 '\n',
 'You',
 ' are']

In [97]:
len(train_ids)

301966

In [101]:
t = random.randrange(len(train_ids - T))

In [102]:
t

80849

In [106]:
vocab_size

50257